In [181]:
import pandas as pd
DORI_ONC = pd.read_csv("/data/DORI/DORI-ONC/DORI.csv")
print(DORI_ONC["species_label_clean"].value_counts())
print(DORI_ONC["ecotype_label_clean"].value_counts())
print(DORI_ONC["call_annotation_clean"].value_counts())


species_label_clean
orca                            31329
humpback                        18601
pacific white sided dolphin      2968
fin whale                        1991
sea lion                         1070
false killer whale                244
multiple classes                  235
sperm whale                       126
noise                             111
northern right whale dolphin        7
gray                                7
minke                               4
uncertain                           1
Name: count, dtype: int64
ecotype_label_clean
srkw                12352
transient            2873
offshore              184
nrkw                   41
multiple classes        3
noise                   1
Name: count, dtype: int64
call_annotation_clean
clicks        4049
uncertain     1603
s1            1216
orca           989
transient      843
              ... 
s44,s37          1
s4,s18,s12       1
s18,s36          1
s1,s19           1
s3,s1            1
Name: count, Length: 262, d

# Missing data:
About half of the data is not included in the DORI data set, only embeddings are provided

In [182]:
# Check which files listed in the shared DORI.csv manifest are missing on disk,
# broken down per repo (the manifest is identical across all four DORI-* repos,
# but the audio is split across their individual data/train_*/test folders)
from pathlib import Path

DORI_ROOT = Path("/data/DORI")
DATA_DIRS = [
    DORI_ROOT / "DORI-ONC",
    DORI_ROOT / "DORI-OOI",
    DORI_ROOT / "DORI-Orcasound",
    DORI_ROOT / "DORI-SanctSound",
]

manifest_files = DORI_ONC["filename"].apply(lambda f: Path(f).name)

existing_files = set()
for d in DATA_DIRS:
    repo_files = {
        p.name
        for p in d.rglob("*")
        if p.is_file() and p.suffix.lower() in (".flac", ".wav", ".mp3")
    }
    existing_files |= repo_files
    missing_mask = ~manifest_files.isin(repo_files)
    print(f"{d.name}: on_disk={len(repo_files)} missing={missing_mask.sum()} ({missing_mask.mean():.1%})")

DORI-ONC: on_disk=32247 missing=34891 (58.7%)
DORI-OOI: on_disk=1356 missing=59090 (99.4%)
DORI-Orcasound: on_disk=1585 missing=57665 (97.0%)
DORI-SanctSound: on_disk=0 missing=59442 (100.0%)


# Verify that the missing data is not present on HF 

In [183]:
# The check above only compares the manifest against local disk, which conflates
# two very different things: files that were never released as raw audio on HF
# (e.g. DORI-SanctSound only ships whisper embeddings, no .flac) vs files that
# genuinely failed to download. Query the HF repos directly to tell them apart.
# Requires: pip install huggingface_hub  (available in thalios_pipeline/.venv)
from huggingface_hub import HfApi

api = HfApi()
HF_ORG = "DORI-SRKW"
REPOS = ["DORI-ONC", "DORI-OOI", "DORI-Orcasound", "DORI-SanctSound"]

for repo in REPOS:
    hf_files = api.list_repo_files(f"{HF_ORG}/{repo}", repo_type="dataset")
    hf_audio = {Path(f).name for f in hf_files if f.lower().endswith((".flac", ".wav", ".mp3"))}

    local_dir = DORI_ROOT / repo
    local_audio = {
        p.name for p in local_dir.rglob("*")
        if p.is_file() and p.suffix.lower() in (".flac", ".wav", ".mp3")
    }

    missing = hf_audio - local_audio  # on HF, not downloaded -> real gap
    unexpected = local_audio - hf_audio  # on disk, not on HF -> shouldn't happen

    print(f"{repo}: on_hf={len(hf_audio)} local={len(local_audio)} "
          f"not_downloaded={len(missing)} local_only={len(unexpected)}")


DORI-ONC: on_hf=32247 local=32247 not_downloaded=0 local_only=0
DORI-OOI: on_hf=1356 local=1356 not_downloaded=0 local_only=0
DORI-Orcasound: on_hf=1585 local=1585 not_downloaded=0 local_only=0
DORI-SanctSound: on_hf=0 local=0 not_downloaded=0 local_only=0


# DCLDE Dataset

I am going to now load the DCLDE dataset - I hope that it will be possible to see where the DCLDE and DORI datasets overlap.

In [184]:
DCLDE = pd.read_csv("/home/noah/HALLO_encoder_collection/data_raw/DCLDE_2027/20260921_131510/annotations.csv")
print(DCLDE['ClassSpecies'].value_counts())
print(DCLDE['CallType'].value_counts())

/tmp/ipykernel_173886/3061012766.py:1: DtypeWarning: Columns (5,7,10,12,13,15,16,25) have mixed types. Specify dtype option on import or set low_memory=False.
  DCLDE = pd.read_csv("/home/noah/HALLO_encoder_collection/data_raw/DCLDE_2027/20260921_131510/annotations.csv")


ClassSpecies
HW        124977
SRKW       20890
TKW        12707
UndBio     11820
AB         11623
NRKW        8266
SAR         8078
KW_und      6450
OKW         2698
Name: count, dtype: int64
CallType
BP                  876
S04                 781
S01                 598
Unk                 371
S19                 220
                   ... 
S44 or S02i?          1
S02iii & Unk          1
Unk & S41? & S36      1
S16/17?               1
Groan and knock       1
Name: count, Length: 155, dtype: int64


# Are DORI annotations file-level or segment-level?

Check the `segment_start`/`segment_end` columns: non-null values indicate a time-localized
event within the file (segment-level); null values indicate the label applies to the whole
clip with no localization (file-level).

In [185]:
DORI_LOCAL = DORI_ONC[DORI_ONC['filename'].apply(lambda f: Path(f).name).isin(existing_files)]

print('[local] segment_start/segment_end null counts:')
print(DORI_LOCAL[['segment_start', 'segment_end']].isna().sum())
print()

print((DORI_LOCAL['segment_start'].isna() & DORI_LOCAL['segment_end'].isna()).sum())

[local] segment_start/segment_end null counts:
segment_start    1963
segment_end      3823
dtype: int64

1963


In [186]:
segment_rows = DORI_LOCAL[~(DORI_LOCAL['segment_start'].isna() & DORI_LOCAL['segment_end'].isna())]
print('[local] total segment-level annotations:', len(segment_rows), '/', len(DORI_LOCAL))
print(segment_rows.describe())
dclde_stems = set(DCLDE['Soundfile'].apply(lambda f: Path(f).stem if pd.notna(f) else None))
overlap_mask = segment_rows['filename'].apply(lambda f: Path(f).stem).isin(dclde_stems)

print('[local] segment-level annotations overlapping with DCLDE:', overlap_mask.sum(), '/', len(segment_rows))

[local] total segment-level annotations: 24717 / 26680
         Unnamed: 0           lat           lon         depth  segment_start  \
count  24717.000000  24564.000000  24564.000000  23202.000000   24717.000000   
mean    7743.440507     48.814055   -124.058311    386.141716     122.474926   
std     5391.545919      0.605692      3.660818    561.111787      94.617210   
min       12.000000     44.637100   -130.430694      7.000000    -120.000000   
25%     3379.000000     48.813685   -125.280108    143.000000      31.200000   
50%     6675.000000     49.042645   -123.317755    168.000000     105.000000   
75%    11708.000000     49.043300   -123.317440    172.000000     210.000000   
max    41087.000000     54.258806    -53.121256   2670.000000     297.600000   

        segment_end  
count  22857.000000  
mean     135.474157  
std       94.797160  
min        1.200000  
25%       45.600000  
50%      120.000000  
75%      225.000000  
max      300.000000  
[local] segment-level anno

# Do embeddings account for the missing files?

Each repo (except DORI-ONC) ships a `whisper-tiny_embeddings_*.parquet` with a `filename` column.
For DORI-OOI and DORI-SanctSound this matches the raw audio naming exactly, so missing files can
be joined directly. DORI-Orcasound's embeddings use a different naming scheme (`OS_<date>_seg_<n>`)
than its raw audio (`rpi-<station>_<timestamp>`), so exact-name matching won't find overlap there.

In [187]:
import pyarrow.parquet as pq

EMBEDDING_PATHS = {
    "DORI-OOI": DORI_ROOT / "DORI-OOI/whisper-tiny_embeddings_OOI.parquet",
    "DORI-Orcasound": DORI_ROOT / "DORI-Orcasound/whisper-tiny_embeddings_OrcaSound.parquet",
    "DORI-SanctSound": DORI_ROOT / "DORI-SanctSound/whisper-tiny_embeddings_SanctSound.parquet",
}

for d in DATA_DIRS:
    repo_files = {
        p.name
        for p in d.rglob("*")
        if p.is_file() and p.suffix.lower() in (".flac", ".wav", ".mp3")
    }
    missing_names = set(manifest_files[~manifest_files.isin(repo_files)])

    emb_path = EMBEDDING_PATHS.get(d.name)
    if emb_path is None or not emb_path.exists():
        print(f"{d.name}: missing={len(missing_names)} | no embeddings parquet")
        continue

    emb_names = set(pq.read_table(emb_path, columns=["filename"]).to_pandas()["filename"])
    covered = missing_names & emb_names
    print(f"{d.name}: missing={len(missing_names)} | covered_by_embeddings={len(covered)} ({len(covered)/len(missing_names):.1%})")

DORI-ONC: missing=34337 | no embeddings parquet
DORI-OOI: missing=58121 | covered_by_embeddings=491 (0.8%)
DORI-Orcasound: missing=56891 | covered_by_embeddings=0 (0.0%)
DORI-SanctSound: missing=58473 | covered_by_embeddings=31816 (54.4%)


# Holdout split:
We want to include three different deployments for our test set with the combined DORI/DCLDE Dataset
- Simres stations on Saturna (DCLDE Simres eastpoint DS)
- SMRU station on line kilm (DCLDE smru lime-kiln)
- ONC VENUS node ()



In [188]:
DORI_ONC.columns

Index(['Unnamed: 0', 'filename', 'license', 'lat', 'lon', 'depth', 'date',
       'doi', 'species_label', 'species_label_clean', 'species_label_source',
       'ecotype_label_clean', 'ecotype_label_source', 'segment_start',
       'segment_end', 'call_annotation', 'call_annotation_clean',
       'call_annotation_source', 'sighting_annotation',
       'sighting_annotation_source'],
      dtype='object')

# DORI-ONC: unique DOIs and locations

No station-name column exists, so "location" is taken as unique `(lat, lon)` pairs. Exact
float pairs can differ slightly for the same physical station, so also check locations
rounded to 3 decimal places (~110m) to see how much that collapses.

In [189]:
print(f"Unique DOIs: {DORI_ONC['doi'].nunique()} (missing on {DORI_ONC['doi'].isna().sum()} rows)")
print(DORI_ONC['doi'].value_counts(dropna=False).head(20))
print()

locations = DORI_ONC[['lat', 'lon']].dropna().drop_duplicates()
locations_rounded = DORI_ONC[['lat', 'lon']].dropna().round(3).drop_duplicates()
print(f"Unique locations (exact lat/lon): {len(locations)} (missing on {DORI_ONC['lat'].isna().sum()} rows)")
print(f"Unique locations (rounded to 3dp): {len(locations_rounded)}")
locations.sort_values(['lat', 'lon'])

Unique DOIs: 243 (missing on 3028 rows)
doi
https://doi.org/10.25921/saca-sp25                                                                           31836
NaN                                                                                                           3028
https://data.oceannetworks.ca/DatasetLandingPage?doidataset=10.34943/6cfdfb5a-b8d1-43e3-8bb1-bf2ae31947e6     1588
https://doi.org/10.34943/e71a2f1d-2dd2-4b20-bebd-952625801a95                                                 1333
https://doi.org/10.34943/ee5101e3-6454-453a-ba74-6d42fd832850                                                 1163
https://doi.org/10.34943/47922d0b-240b-4f16-8cd2-f52b694dee5b                                                 1073
https://doi.org/10.34943/de7c4377-f3df-4445-a651-95eb593cf3a3                                                  967
https://data.oceannetworks.ca/DatasetLandingPage?doidataset=10.34943/8aa1dc24-2e86-45ad-a46b-6ed1c9a3e0df      905
https://data.oceannetworks.ca/Datase

,lat,lon
24647,-125.522200,47.765100
24985,44.369500,-124.954000
24986,44.637100,-124.306000
24123,47.316732,-127.050142
57763,47.317500,-124.414600
...,...,...
8400,50.020760,-125.235400
7858,50.020767,-125.235367
1212,50.020767,-125.235350
8407,50.021000,-125.235000


## Is DORI-ONC just the VENUS node?

No — it blends multiple ONC observatories. Depth is the giveaway: VENUS (Strait of Georgia)
tops out around 300m, while several location clusters here go to 650-2670m, matching NEPTUNE's
offshore Vancouver Island sites (Barkley Canyon, Cascadia Basin). A few rows also land off
Oregon/Newfoundland or have swapped lat/lon — flagged separately, not part of either observatory.

Bucketing below is a geographic/depth heuristic (bounding boxes for the Strait of Georgia vs.
the NEPTUNE offshore corridor), not an authoritative ONC site lookup.

In [190]:
import numpy as np

NAMED_SITES = [
    {'name': 'Strait of Georgia East Node', 'lat': 49.042835, 'lon': -123.317265},
    {'name': 'Strait of Georgia Central Node', 'lat': 49.040437, 'lon': -123.425795},
    {'name': 'Saanich Inlet Central Node', 'lat': 48.650900, 'lon': -123.486712},
]
VENUS_RADIUS_M = 1000

def haversine_m(lat1, lon1, lat2, lon2):
    R = 6371000.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dlambda / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

loc_rows = DORI_ONC.dropna(subset=['lat', 'lon']).copy()
loc_rows['lat_r'] = loc_rows['lat'].round(3)
loc_rows['lon_r'] = loc_rows['lon'].round(3)

# For every distinct (lat_r, lon_r), find the nearest named site and its distance
_clusters_for_radius = loc_rows.groupby(['lat_r', 'lon_r']).size().reset_index(name='rows')
_dists = pd.DataFrame({s['name']: haversine_m(_clusters_for_radius['lat_r'], _clusters_for_radius['lon_r'], s['lat'], s['lon']) for s in NAMED_SITES})
_clusters_for_radius['min_dist'] = _dists.min(axis=1)
_clusters_for_radius['nearest_site'] = _dists.idxmin(axis=1)

SITE_BY_COORD = dict(
    zip(
        map(tuple, _clusters_for_radius.loc[_clusters_for_radius['min_dist'] <= VENUS_RADIUS_M, ['lat_r', 'lon_r']].values),
        _clusters_for_radius.loc[_clusters_for_radius['min_dist'] <= VENUS_RADIUS_M, 'nearest_site'],
    )
)
VENUS_COORDS = set(SITE_BY_COORD.keys())

def classify_site(lat, lon):
    nearest = SITE_BY_COORD.get((round(lat, 3), round(lon, 3)))
    if nearest is None:
        return 'other hydrophone'
    return 'VENUS (Saanich Inlet)' if 'Saanich' in nearest else 'VENUS (Strait of Georgia)'

site_clusters = (
    loc_rows.groupby(['lat_r', 'lon_r'])
    .agg(rows=('filename', 'size'), depth_mean=('depth', 'mean'))
    .reset_index()
)
site_clusters['site'] = site_clusters.apply(lambda r: classify_site(r['lat_r'], r['lon_r']), axis=1)

print(site_clusters.groupby('site').agg(row_count=('rows', 'sum'), num_locations=('rows', 'count')).sort_values('row_count', ascending=False))
print()
site_clusters.sort_values('rows', ascending=False)

                           row_count  num_locations
site                                               
other hydrophone               45152             46
VENUS (Strait of Georgia)      14023              9
VENUS (Saanich Inlet)             30              3



,lat_r,lon_r,rows,depth_mean,site
14,48.038,-124.728,29943,31.000000,other hydrophone
47,49.043,-123.316,5280,163.868656,VENUS (Strait of Georgia)
45,49.043,-123.318,4367,169.793127,VENUS (Strait of Georgia)
46,49.043,-123.317,2316,167.960276,VENUS (Strait of Georgia)
24,48.490,-125.004,1964,75.888442,other hydrophone
35,48.814,-125.282,1818,100.851485,other hydrophone
32,48.699,-126.872,1662,1315.000000,other hydrophone
41,49.040,-123.425,1114,297.000000,VENUS (Strait of Georgia)
48,49.081,-123.341,1113,143.000000,other hydrophone
38,48.814,-125.275,1027,95.000000,other hydrophone


In [191]:
n_venus = loc_rows[['lat_r', 'lon_r']].apply(tuple, axis=1).isin(VENUS_COORDS).sum()
print(f"{n_venus} / {len(loc_rows)}")

14053 / 59205


In [192]:
def classify_provider(lat, lon):
    nearest = SITE_BY_COORD.get((round(lat, 3), round(lon, 3))) if pd.notna(lat) and pd.notna(lon) else None
    if nearest is None:
        return 'ONC (DORI)'
    return 'ONC VENUS (Saanich)' if 'Saanich' in nearest else 'ONC VENUS (Strait of Georgia)'

DORI_ONC['provider'] = DORI_ONC.apply(lambda r: classify_provider(r['lat'], r['lon']), axis=1)
DORI_ONC['provider'].value_counts()

provider
ONC (DORI)                       45389
ONC VENUS (Strait of Georgia)    14023
ONC VENUS (Saanich)                 30
Name: count, dtype: int64

In [193]:
depth_by_loc = loc_rows.dropna(subset=['depth']).groupby(['lat_r', 'lon_r'])['depth'].agg(['nunique', 'min', 'max', 'count']).reset_index()
depth_by_loc['range'] = depth_by_loc['max'] - depth_by_loc['min']
varying = depth_by_loc[depth_by_loc['nunique'] > 1].sort_values('range', ascending=False)

print(f"Locations: {len(depth_by_loc)} | with >1 distinct depth: {len(varying)}")
varying

Locations: 54 | with >1 distinct depth: 17


,lat_r,lon_r,nunique,min,max,count,range
21,48.490,-125.004,4,-90.500,90.5,1964,181.000
41,49.043,-123.318,5,150.000,172.0,4365,22.000
18,48.394,-124.653,2,-10.500,10.5,157,21.000
31,48.814,-125.282,3,94.000,105.0,1818,11.000
46,49.081,-123.339,3,141.000,147.0,1014,6.000
11,47.924,-129.109,2,2273.000,2277.0,634,4.000
50,49.154,-124.802,2,107.000,111.0,4,4.000
24,48.652,-123.486,3,91.000,95.0,16,4.000
32,48.814,-125.281,2,96.209,100.0,65,3.791
52,50.021,-125.235,5,7.000,10.0,167,3.000


In [194]:
# site_clusters['depth'].value_counts()
site_clusters[site_clusters['site'] == 'VENUS (Strait of Georgia)']


,lat_r,lon_r,rows,depth_mean,site
39,49.039,-123.425,493,298.000000,VENUS (Strait of Georgia)
40,49.040,-123.426,303,298.267327,VENUS (Strait of Georgia)
41,49.040,-123.425,1114,297.000000,VENUS (Strait of Georgia)
42,49.041,-123.426,58,298.000000,VENUS (Strait of Georgia)
43,49.042,-123.318,15,170.000000,VENUS (Strait of Georgia)
44,49.042,-123.317,77,169.831169,VENUS (Strait of Georgia)
45,49.043,-123.318,4367,169.793127,VENUS (Strait of Georgia)
46,49.043,-123.317,2316,167.960276,VENUS (Strait of Georgia)
47,49.043,-123.316,5280,163.868656,VENUS (Strait of Georgia)


In [195]:
import folium
import numpy as np
from folium import Element

# Requires: uv pip install --python <venv> folium  (adds folium, branca, xyzservices)
# CartoDB tiles now require an API key, so this uses keyless OpenStreetMap tiles.

# Official ONC station names/coords, from the ONC site listing (not derivable from DORI.csv,
# which has no station-name column) - shown as separate star markers.
NAMED_SITES = [
    {'name': 'Strait of Georgia East Node', 'lat': 49.042835, 'lon': -123.317265, 'depth': 170},
    {'name': 'Strait of Georgia Central Node', 'lat': 49.040437, 'lon': -123.425795, 'depth': 300},
    {'name': 'Saanich Inlet Central Node', 'lat': 48.650900, 'lon': -123.486712, 'depth': 100},
]

map_rows = loc_rows.copy()
map_rows['site'] = map_rows.apply(lambda r: classify_site(r['lat'], r['lon']), axis=1)

# Drop rows far outside BC (off Newfoundland, off Oregon) and swapped lat/lon,
# which otherwise force the map to zoom out to the whole continent.
valid = (map_rows['lat'].between(46, 60)) & (map_rows['lon'].between(-135, -115))
n_excluded = (~valid).sum()
map_rows = map_rows[valid]

clusters = (
    map_rows.groupby(['lat_r', 'lon_r', 'site']).size().reset_index(name='rows')
)

PALETTE = {
    'VENUS (Strait of Georgia)': '#eb6834',
    'VENUS (Saanich Inlet)': '#1baf7a',
    'other hydrophone': '#2a78d6',
}
INK = '#0b0b0b'

m = folium.Map(tiles='OpenStreetMap', control_scale=True)

for site, color in PALETTE.items():
    sub = clusters[clusters['site'] == site]
    fg = folium.FeatureGroup(name=f"{site} (n={sub['rows'].sum()})")
    for _, r in sub.iterrows():
        radius = float(np.clip(5 + 1.5 * np.log1p(r['rows']), 5, 15))
        folium.CircleMarker(
            location=(r['lat_r'], r['lon_r']),
            radius=radius,
            color='#fcfcfb', weight=1, fill=True, fill_color=color, fill_opacity=0.85,
            popup=f"{site}<br>{r['lat_r']}, {r['lon_r']}<br>{r['rows']} annotation rows",
        ).add_to(fg)
    fg.add_to(m)

named_fg = folium.FeatureGroup(name='Official ONC site names')
for s in NAMED_SITES:
    folium.Circle(
        location=(s['lat'], s['lon']),
        radius=1000,  # meters
        color='#8b1a1a', weight=1.5, dash_array='4 4', fill=True, fill_color='#8b1a1a', fill_opacity=0.06,
    ).add_to(named_fg)
    folium.Marker(
        location=(s['lat'], s['lon']),
        icon=folium.Icon(color='darkred', icon='star', prefix='fa'),
        tooltip=s['name'],
        popup=f"<b>{s['name']}</b><br>{s['lat']}, {s['lon']}<br>depth {s['depth']}m",
    ).add_to(named_fg)
named_fg.add_to(m)

bounds = [[clusters['lat_r'].min() - 0.3, clusters['lon_r'].min() - 0.3],
          [clusters['lat_r'].max() + 0.3, clusters['lon_r'].max() + 0.3]]
m.fit_bounds(bounds)

legend_html = f'''
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 9999;
     background: #fcfcfb; padding: 10px 14px; border-radius: 6px;
     border: 1px solid #c3c2b7; font-family: sans-serif; font-size: 13px; color: {INK};">
  <b>DORI-ONC hydrophones</b><br>
  {''.join(f'<span style="display:inline-block;width:10px;height:10px;border-radius:50%;background:{c};margin-right:6px;"></span>{s}<br>' for s, c in PALETTE.items())}
  <span style="display:inline-block;width:10px;">&#9733;</span>Official ONC site name (dashed ring = 1km radius)<br>
  <span style="color:#898781;">{n_excluded} out-of-region/invalid rows excluded</span>
</div>
'''
m.get_root().html.add_child(Element(legend_html))
folium.LayerControl(collapsed=False).add_to(m)

m

The VENUS deployment contains two nodes in the straight of georgia and one node in the Saanich Inlet Central

In [196]:
DORI_ONC['provider']

0        ONC (DORI)
1        ONC (DORI)
2        ONC (DORI)
3        ONC (DORI)
4        ONC (DORI)
            ...    
59437    ONC (DORI)
59438    ONC (DORI)
59439    ONC (DORI)
59440    ONC (DORI)
59441    ONC (DORI)
Name: provider, Length: 59442, dtype: object

In [197]:
print(DORI_ONC['provider'].value_counts())
print(DORI_ONC['species_label_source'].value_counts())
print(DORI_ONC['ecotype_label_source'].value_counts())
print("human generated")
print(DORI_ONC[DORI_ONC['ecotype_label_source'] == 'DORI (human-generated)']['ecotype_label_clean'].value_counts())
print("pseudo-label")
DORI_ONC[DORI_ONC['ecotype_label_source'] == 'DORI (pseudo-label)']['ecotype_label_clean'].value_counts()

provider
ONC (DORI)                       45389
ONC VENUS (Strait of Georgia)    14023
ONC VENUS (Saanich)                 30
Name: count, dtype: int64
species_label_source
SanctSound                31394
DORI (human-generated)    25522
DORI (pseudo-label)        1942
ONC                         446
Name: count, dtype: int64
ecotype_label_source
DORI (human-generated)    8971
DORI (pseudo-label)       6436
ONC                         45
offshore                     1
Name: count, dtype: int64
human generated
ecotype_label_clean
srkw                6823
transient           1998
offshore             129
nrkw                  19
noise                  1
multiple classes       1
Name: count, dtype: int64
pseudo-label


ecotype_label_clean
srkw                5502
transient            860
offshore              51
nrkw                  21
multiple classes       2
Name: count, dtype: int64

There are 6823 human generated srkw labels and 5502 pseudo-labels for srkw

In [198]:
print("human generated")
print(DORI_ONC[DORI_ONC['call_annotation_clean'] != 'clicks'][DORI_ONC['ecotype_label_source'] == 'DORI (human-generated)']['ecotype_label_clean'].value_counts())
print("pseudo-label")
print(DORI_ONC[DORI_ONC['call_annotation_clean'] != 'clicks'][DORI_ONC['ecotype_label_source'] == 'DORI (pseudo-label)']['ecotype_label_clean'].value_counts())
# DORI_ONC['call_annotation_clean'].value_counts()

human generated
ecotype_label_clean
srkw                6365
transient           1989
offshore             129
nrkw                  18
noise                  1
multiple classes       1
Name: count, dtype: int64
pseudo-label
ecotype_label_clean
srkw                3327
transient            842
offshore              51
nrkw                  20
multiple classes       2
Name: count, dtype: int64


/tmp/ipykernel_173886/278929390.py:2: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  print(DORI_ONC[DORI_ONC['call_annotation_clean'] != 'clicks'][DORI_ONC['ecotype_label_source'] == 'DORI (human-generated)']['ecotype_label_clean'].value_counts())
/tmp/ipykernel_173886/278929390.py:4: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  print(DORI_ONC[DORI_ONC['call_annotation_clean'] != 'clicks'][DORI_ONC['ecotype_label_source'] == 'DORI (pseudo-label)']['ecotype_label_clean'].value_counts())


luckily there is still a good amount which isnt represented by clicks in the labels: 6365 human generated and 3327 pseudo labels

In [199]:
print("human generated")
print(DORI_ONC[DORI_ONC['provider'].isin(["ONC VENUS (Strait of Georgia)", "ONC VENUS (Saanich)"])][DORI_ONC['call_annotation_clean'] != 'clicks'][DORI_ONC['ecotype_label_source'] == 'DORI (human-generated)']['ecotype_label_clean'].value_counts())
print("pseudo-label")
print(DORI_ONC[DORI_ONC['provider'].isin(["ONC VENUS (Strait of Georgia)", "ONC VENUS (Saanich)"])][DORI_ONC['call_annotation_clean'] != 'clicks'][DORI_ONC['ecotype_label_source'] == 'DORI (pseudo-label)']['ecotype_label_clean'].value_counts())

human generated
ecotype_label_clean
srkw         3907
transient    1200
noise           1
Name: count, dtype: int64
pseudo-label
ecotype_label_clean
srkw         1477
transient     457
nrkw            1
Name: count, dtype: int64


/tmp/ipykernel_173886/2419869312.py:2: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  print(DORI_ONC[DORI_ONC['provider'].isin(["ONC VENUS (Strait of Georgia)", "ONC VENUS (Saanich)"])][DORI_ONC['call_annotation_clean'] != 'clicks'][DORI_ONC['ecotype_label_source'] == 'DORI (human-generated)']['ecotype_label_clean'].value_counts())
/tmp/ipykernel_173886/2419869312.py:2: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  print(DORI_ONC[DORI_ONC['provider'].isin(["ONC VENUS (Strait of Georgia)", "ONC VENUS (Saanich)"])][DORI_ONC['call_annotation_clean'] != 'clicks'][DORI_ONC['ecotype_label_source'] == 'DORI (human-generated)']['ecotype_label_clean'].value_counts())
/tmp/ipykernel_173886/2419869312.py:4: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  print(DORI_ONC[DORI_ONC['provider'].isin(["ONC VENUS (Strait of Georgia)", "ONC VENUS (Saanich)"])][DORI_ONC['call_annotation_clean'] != 'clicks'][DORI_O

Once again checking that the provider is from ONC VENUS Node reduces the models down to 3907 for human generated and 1477 for pseudo labels

In [200]:
print(DORI_ONC[DORI_ONC['provider'].isin(["ONC VENUS (Strait of Georgia)", "ONC VENUS (Saanich)"])][DORI_ONC['call_annotation_clean'] != 'clicks'][DORI_ONC['species_label_source'] == 'DORI (human-generated)']['species_label_clean'].value_counts())

species_label_clean
orca                           5977
humpback                       2092
pacific white sided dolphin      45
sea lion                          2
noise                             1
Name: count, dtype: int64


/tmp/ipykernel_173886/2413199282.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  print(DORI_ONC[DORI_ONC['provider'].isin(["ONC VENUS (Strait of Georgia)", "ONC VENUS (Saanich)"])][DORI_ONC['call_annotation_clean'] != 'clicks'][DORI_ONC['species_label_source'] == 'DORI (human-generated)']['species_label_clean'].value_counts())
/tmp/ipykernel_173886/2413199282.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  print(DORI_ONC[DORI_ONC['provider'].isin(["ONC VENUS (Strait of Georgia)", "ONC VENUS (Saanich)"])][DORI_ONC['call_annotation_clean'] != 'clicks'][DORI_ONC['species_label_source'] == 'DORI (human-generated)']['species_label_clean'].value_counts())


For the purposes of this set fitting into the larger DS we can treat the other classes: pacific white sided dolphin and sea lion as Unk_Bio

In [201]:
print(DORI_ONC[~DORI_ONC['provider'].isin(["ONC VENUS (Strait of Georgia)", "ONC VENUS (Saanich)"])][DORI_ONC['call_annotation_clean'] != 'clicks'][DORI_ONC['species_label_source'] == 'DORI (human-generated)']['species_label_clean'].value_counts())

species_label_clean
orca                            4794
pacific white sided dolphin     2895
humpback                        2004
sea lion                        1046
false killer whale               244
multiple classes                 235
noise                            110
sperm whale                       67
gray                               6
minke                              4
northern right whale dolphin       3
uncertain                          1
Name: count, dtype: int64


/tmp/ipykernel_173886/3692605177.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  print(DORI_ONC[~DORI_ONC['provider'].isin(["ONC VENUS (Strait of Georgia)", "ONC VENUS (Saanich)"])][DORI_ONC['call_annotation_clean'] != 'clicks'][DORI_ONC['species_label_source'] == 'DORI (human-generated)']['species_label_clean'].value_counts())
/tmp/ipykernel_173886/3692605177.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  print(DORI_ONC[~DORI_ONC['provider'].isin(["ONC VENUS (Strait of Georgia)", "ONC VENUS (Saanich)"])][DORI_ONC['call_annotation_clean'] != 'clicks'][DORI_ONC['species_label_source'] == 'DORI (human-generated)']['species_label_clean'].value_counts())


In the non- VENUS deployment there are quite a bit of diversity in the annotations containing many biotic calls 

In [202]:
print(DCLDE.columns)
print(DORI_ONC.columns)
DORI_ONC['sighting_annotation_source'].value_counts()
print(DCLDE['UTC'])
print(DORI_ONC['date'])

print("DORI")
print(DORI_ONC['filename'])
print("DCLDE")
print(DCLDE['Soundfile'])

Index(['Soundfile', 'Dataset', 'LowFreqHz', 'HighFreqHz', 'FileEndSec', 'UTC',
       'FileBeginSec', 'ClassSpecies', 'KW', 'KW_certain', 'Ecotype',
       'Provider', 'AnnotationLevel', 'FilePath', 'FileOk', 'CallType',
       'CalltypeCategory', 'HasQ', 'CalltypeHasQ', 'LocalPath', 'LocalFileOk',
       'CenterTime', 'Duration', 'EcotypeCertain', 'Labels',
       'BackgroundMethod', 'uid'],
      dtype='object')
Index(['Unnamed: 0', 'filename', 'license', 'lat', 'lon', 'depth', 'date',
       'doi', 'species_label', 'species_label_clean', 'species_label_source',
       'ecotype_label_clean', 'ecotype_label_source', 'segment_start',
       'segment_end', 'call_annotation', 'call_annotation_clean',
       'call_annotation_source', 'sighting_annotation',
       'sighting_annotation_source', 'provider'],
      dtype='object')
0          2013-10-11 00:10:11.439
1              2013-10-11 00:10:28
2            2013-10-11 00:10:57.6
3          2013-10-11 00:21:26.176
4          2013-10-11 00

In [203]:
print(DCLDE['FileEndSec'].value_counts())
print(DORI_ONC['segment_end'].value_counts())

print(DCLDE['FileBeginSec'].value_counts())
print(DORI_ONC['segment_start'].value_counts())

FileEndSec
55.0      8286
57.5      8278
52.5      8275
50.0      8259
47.5      8243
          ... 
8465.0       1
8467.5       1
8550.0       1
8552.5       1
6775.0       1
Name: count, Length: 107123, dtype: int64
segment_end
15.00000     2897
60.00000     1742
300.00000    1279
30.00000     1221
45.00000      900
             ... 
232.80000       1
222.60000       1
32.91210        1
33.11289        1
32.40000        1
Name: count, Length: 436, dtype: int64
FileBeginSec
0.0       8344
50.0      8287
52.5      8278
47.5      8271
45.0      8258
          ... 
8515.0       1
8517.5       1
8520.0       1
8522.5       1
8525.0       1
Name: count, Length: 92983, dtype: int64
segment_start
0.000000      33777
285.000000     2330
30.000000      1349
270.000000     1109
15.000000      1070
              ...  
44.000000         1
30.614817         1
29.400000         1
31.200000         1
31.000000         1
Name: count, Length: 436, dtype: int64


In [204]:
print(DCLDE['Dataset'].value_counts())
print(DCLDE['Provider'].value_counts())
print(DORI_ONC['provider'].value_counts())
print(DORI_ONC['provider'].apply(lambda x: x.replace("ONC", "DORI").replace("(DORI)", "")).value_counts())

Dataset
WVanIsl            448951
NorthBc            297702
BarkleyCanyon      130112
HaroStraitNorth     84383
HaroStraitSouth     80660
SwanChan            18569
Cpe_Elz             16149
CarmanahPt          12393
StrGeoS2            11836
Quin_Can            11375
Tekteksen           10973
StrGeoN1            10517
Field_HTI            7131
orcasound_lab        6612
StrGeoS1             6053
BoundaryPass         4584
bush_point           3510
StraitofGeorgia      3292
MS_671879205         2473
Field_SondTrap       2276
LimeKiln             2186
port_townsend        1923
KB_67424266          1826
StrGeoN2             1783
HE_67424266           949
KB_67383303           740
MS_6897               394
MS_5360               365
HE_67391498           148
RB_335826997          140
KB_5354               106
RB_67424266            76
Name: count, dtype: int64
Provider
DFO_CRP           746653
JASCO_VFPA        169627
ONC               130112
DFO_WDLP           61151
SIO                27524


In [205]:
print(DORI_ONC.columns)
print(DCLDE.columns)
print(DORI_ONC['species_label_clean'].value_counts())
print(DORI_ONC['ecotype_label_clean'].value_counts())

print(DORI_ONC['ecotype_label_clean'].value_counts())
print(DCLDE['ClassSpecies'].value_counts())



Index(['Unnamed: 0', 'filename', 'license', 'lat', 'lon', 'depth', 'date',
       'doi', 'species_label', 'species_label_clean', 'species_label_source',
       'ecotype_label_clean', 'ecotype_label_source', 'segment_start',
       'segment_end', 'call_annotation', 'call_annotation_clean',
       'call_annotation_source', 'sighting_annotation',
       'sighting_annotation_source', 'provider'],
      dtype='object')
Index(['Soundfile', 'Dataset', 'LowFreqHz', 'HighFreqHz', 'FileEndSec', 'UTC',
       'FileBeginSec', 'ClassSpecies', 'KW', 'KW_certain', 'Ecotype',
       'Provider', 'AnnotationLevel', 'FilePath', 'FileOk', 'CallType',
       'CalltypeCategory', 'HasQ', 'CalltypeHasQ', 'LocalPath', 'LocalFileOk',
       'CenterTime', 'Duration', 'EcotypeCertain', 'Labels',
       'BackgroundMethod', 'uid'],
      dtype='object')
species_label_clean
orca                            31329
humpback                        18601
pacific white sided dolphin      2968
fin whale                     

In [206]:
print(DCLDE['ClassSpecies'].value_counts())
species_rename_mapping = {
    "srkw":"SRKW",
    "humpback":"HW",
    "transient": "TKW",
    "offshore": "OKW",
    "nrkw": "NRKW",
    "orca": "KW_und",
    "noise": "AB",
    "pacific white sided dolphin": "UndBio",
    "fin whale": "UndBio",
    "sea lion": "UndBio",
    "false killer whale": "UndBio",
    "multiple classes": "UndBio",
    "sperm whale": "UndBio",
    "northern right whale dolphin": "UndBio",
    "gray": "UndBio",
    "minke": "UndBio",
    # "uncertain": "UndBio",
}
print(DORI_ONC['ecotype_label_clean'].apply(lambda x: species_rename_mapping[x] if x in species_rename_mapping else x).value_counts())

ClassSpecies
HW        124977
SRKW       20890
TKW        12707
UndBio     11820
AB         11623
NRKW        8266
SAR         8078
KW_und      6450
OKW         2698
Name: count, dtype: int64
ecotype_label_clean
SRKW      12352
TKW        2873
OKW         184
NRKW         41
UndBio        3
AB            1
Name: count, dtype: int64


In [207]:
print(DCLDE['KW'].value_counts())
DCLDE['KW_certain'].value_counts()

KW
0    1121098
1      59089
Name: count, dtype: int64


KW_certain
1.0    57300
0.0      841
Name: count, dtype: int64

In [208]:
print(DCLDE['CallType'].value_counts())
DORI_ONC['call_annotation_clean'].value_counts()

CallType
BP                  876
S04                 781
S01                 598
Unk                 371
S19                 220
                   ... 
S44 or S02i?          1
S02iii & Unk          1
Unk & S41? & S36      1
S16/17?               1
Groan and knock       1
Name: count, Length: 155, dtype: int64


call_annotation_clean
clicks        4049
uncertain     1603
s1            1216
orca           989
transient      843
              ... 
s44,s37          1
s4,s18,s12       1
s18,s36          1
s1,s19           1
s3,s1            1
Name: count, Length: 262, dtype: int64

In [209]:
print(DCLDE['HasQ'].value_counts())

print(DCLDE['CalltypeHasQ'].value_counts())
print(DORI_ONC['species_label_clean'].astype(str).apply(lambda x: 1 if "?" in x else 0).sum())
print(DORI_ONC['ecotype_label_clean'].astype(str).apply(lambda x: 1 if "?" in x else 0).sum())
print(DORI_ONC['call_annotation_clean'].astype(str).apply(lambda x: 1 if "?" in x else 0).sum())

HasQ
False    1180187
Name: count, dtype: int64
CalltypeHasQ
False    1179931
True         256
Name: count, dtype: int64
0
0
0


In [210]:
DCLDE['LocalPath'].value_counts()
from pathlib import Path
root = Path('/data/DORI/DORI-ONC')
import os
stem = lambda f: os.path.splitext(os.path.basename(f))[0]
paths = {stem(f): os.path.join(d, f) for d, _, fs in os.walk(root) for f in fs if f.endswith('.flac')}
DORI_ONC['filename'].map(lambda f: paths.get(stem(f))).value_counts()

filename
/data/DORI/DORI-ONC/train_2014/ICLISTENHF1253_20140124T182322.654Z.flac                5
/data/DORI/DORI-ONC/train_2022/ICLISTENHF1252_20220906T072500.043Z.flac                4
/data/DORI/DORI-ONC/train_2021/JASCOAMARHYDROPHONEE000186_20210910T111305.002Z.flac    3
/data/DORI/DORI-ONC/train_2021/JASCOAMARHYDROPHONEE000186_20210117T232139.363Z.flac    3
/data/DORI/DORI-ONC/train_2022/JASCOAMARHYDROPHONED001025_20221225T142739.436Z.flac    3
                                                                                      ..
/data/DORI/DORI-ONC/train_2019/ICLISTENHF1350_20191213T025103.996Z.flac                1
/data/DORI/DORI-ONC/train_2019/ICLISTENHF1350_20191213T014103.996Z.flac                1
/data/DORI/DORI-ONC/train_2019/ICLISTENHF1350_20191213T011603.996Z.flac                1
/data/DORI/DORI-ONC/train_2019/ICLISTENHF1350_20191213T011103.996Z.flac                1
/data/DORI/DORI-ONC/train_2019/ICLISTENHF1350_20191214T205604.996Z.flac                1
Name: count,

none of the labels have qs so we can skip this for the DORI ds

In [211]:
print(DCLDE['AnnotationLevel'].value_counts())
print(DCLDE.columns)
DORI_ONC.columns

AnnotationLevel
Detection    169282
Call          37605
File            622
Name: count, dtype: int64
Index(['Soundfile', 'Dataset', 'LowFreqHz', 'HighFreqHz', 'FileEndSec', 'UTC',
       'FileBeginSec', 'ClassSpecies', 'KW', 'KW_certain', 'Ecotype',
       'Provider', 'AnnotationLevel', 'FilePath', 'FileOk', 'CallType',
       'CalltypeCategory', 'HasQ', 'CalltypeHasQ', 'LocalPath', 'LocalFileOk',
       'CenterTime', 'Duration', 'EcotypeCertain', 'Labels',
       'BackgroundMethod', 'uid'],
      dtype='object')


Index(['Unnamed: 0', 'filename', 'license', 'lat', 'lon', 'depth', 'date',
       'doi', 'species_label', 'species_label_clean', 'species_label_source',
       'ecotype_label_clean', 'ecotype_label_source', 'segment_start',
       'segment_end', 'call_annotation', 'call_annotation_clean',
       'call_annotation_source', 'sighting_annotation',
       'sighting_annotation_source', 'provider'],
      dtype='object')

In [212]:
DCLDE['EcotypeCertain'].value_counts()
print(DCLDE['Labels'].value_counts())
DCLDE['BackgroundMethod'].value_counts()
DCLDE['CalltypeCategory'].value_counts()

Labels
Background    972678
HW            124977
SRKW           20890
TKW            12707
UndBio         11820
AB             11623
NRKW            8266
SAR             8078
KW_und          6450
OKW             2698
Name: count, dtype: int64


CalltypeCategory
S04    795
S01    706
S16    259
S19    224
S02    211
S44    188
S36    156
S10    152
S42    105
S05     96
S18     67
S03     60
S17     58
S12     30
T01     27
S37     27
S08     23
T02     21
S07     18
S06     13
T04     12
T07     11
T12     11
S33      9
S41      8
S22      7
S40      7
T03      7
S14      6
T13      6
T08      5
T11      5
S31      4
S13      3
S32      1
Name: count, dtype: int64

In [213]:
def select_naive_background_windows(annotations: pd.DataFrame, window_duration: float, hop_duration: float, event_buffer: float) -> pd.DataFrame:
    """The "naive" background_method: slides a window across every annotated
    recording, keeping only starts that don't overlap a labeled event
    (+/- event_buffer). See notebooks/background_windows.ipynb."""
    valid = annotations[annotations["LocalFileOk"]].copy()

    candidates = []
    for local_path, file_rows in tqdm(valid.groupby("LocalPath"), desc="selecting background windows", unit="file"):
        file_duration = librosa.get_duration(path=local_path)
        padded = list(zip(file_rows["FileBeginSec"] - event_buffer, file_rows["FileEndSec"] + event_buffer))

        for start in np.arange(0, file_duration - window_duration, hop_duration):
            end = start + window_duration
            if any(start < pe and end > ps for ps, pe in padded):
                continue
            candidates.append({
                "Soundfile": file_rows["Soundfile"].iloc[0],
                "LocalPath": local_path,
                "Dataset": file_rows["Dataset"].iloc[0],
                "Provider": file_rows["Provider"].iloc[0],
                "FileBeginSec": float(start),
                "FileEndSec": float(end),
            })
    return pd.DataFrame(candidates)

# filename == soundfile
DORI_ONC = DORI_ONC.rename(columns={"filename": "Soundfile"})
# provider col in DORi is Dataset in DCLDE
DORI_ONC = DORI_ONC.rename(columns={"provider": "Dataset"})
DORI_ONC["Dataset"] = DORI_ONC["Dataset"].apply(lambda x: x.replace("ONC", "DORI").replace("(DORI)", ""))
# low / high freq not provided in DORI
DORI_ONC['LowFreqHz'] = pd.NA
DORI_ONC['HighFreqHz'] = pd.NA
# rename other equivelant fields.
DORI_ONC = DORI_ONC.rename(columns={"date": "UTC", "segment_start": 'FileBeginSec', "segment_end": 'FileEndSec'})
# match ClassSpecies logic in DCLDE dataset
DORI_ONC['ecotype_label_clean'] = DORI_ONC['ecotype_label_clean'].fillna(DORI_ONC['species_label_clean'])
# drop uncertain row
DORI_ONC = DORI_ONC[DORI_ONC['ecotype_label_clean'] != "uncertain"]
# clean up ecotype to match DCLDE
species_rename_mapping = {
    "srkw":"SRKW",
    "humpback":"HW",
    "transient": "TKW",
    "offshore": "OKW",
    "nrkw": "NRKW",
    "orca": "KW_und",
    "noise": "AB",
    "pacific white sided dolphin": "UndBio",
    "fin whale": "UndBio",
    "sea lion": "UndBio",
    "false killer whale": "UndBio",
    "multiple classes": "UndBio",
    "sperm whale": "UndBio",
    "northern right whale dolphin": "UndBio",
    "gray": "UndBio",
    "minke": "UndBio",
    # "uncertain": "UndBio",
}
DORI_ONC['ecotype_label_clean'] = DORI_ONC['ecotype_label_clean'].apply(lambda x: species_rename_mapping[x] if x in species_rename_mapping else x)
# rename to 'ClassSpecies'
DORI_ONC = DORI_ONC.rename(columns={'ecotype_label_clean': 'ClassSpecies'})
DORI_ONC['Ecotype'] = DORI_ONC['ClassSpecies'].where(DORI_ONC['ClassSpecies'].isin(DCLDE['Ecotype'].dropna().unique()), pd.NA)
# KW cols
DORI_ONC['KW_certain'] = pd.NA
DORI_ONC['KW'] = DORI_ONC['ClassSpecies'].apply(lambda x: 1 if x in ["KW_und", "SRKW", "TKW", "OKW", "NRKW"] else 0)
# all data currently is from ONC
DORI_ONC["Provider"] = "ONC"
# all annotations considered call level (this might be incorrect)
DORI_ONC['AnnotationLevel'] = 'Call'
# Filepath cols not relevant
DORI_ONC['FilePath'] = pd.NA
DORI_ONC['FileOk'] = True
# calltype
DORI_ONC=DORI_ONC.rename(columns={'call_annotation_clean': 'CallType'})
# not adding calltype category for now.
DORI_ONC['CalltypeCategory'] = pd.NA
# just make everything false as there are no questions in the DS
DORI_ONC['HasQ'] = False
DORI_ONC['CalltypeHasQ'] = False
# localpath / local files
FILE_ROOT = Path('/data/DORI/DORI-ONC')
import os
stem = lambda f: os.path.splitext(os.path.basename(f))[0]
paths = {stem(f): os.path.join(d, f) for d, _, fs in os.walk(FILE_ROOT) for f in fs if f.endswith('.flac')}
DORI_ONC['LocalPath'] = DORI_ONC['Soundfile'].map(lambda f: paths.get(stem(f)))
DORI_ONC['LocalFileOk'] = DORI_ONC['LocalPath'].notna()
# centertime and duration:
DORI_ONC['CenterTime'] = (DORI_ONC['FileEndSec'] + DORI_ONC['FileBeginSec']) / 2
DORI_ONC['Duration'] = DORI_ONC['FileEndSec'] - DORI_ONC['FileBeginSec']
DORI_ONC['EcotypeCertain'] = pd.NA
# labels, same idea as derive_labels in the DCLDE build script
DORI_ONC['Labels'] = DORI_ONC['ClassSpecies']
DORI_ONC['BackgroundMethod'] = pd.NA
# background windows: "naive" method, using select_naive_background_windows defined at the top of this cell
import librosa
import numpy as np
from tqdm import tqdm
# file-level annotations (no segment times) cover the whole file, so those files give no background
events = DORI_ONC.fillna({'FileBeginSec': 0, 'FileEndSec': float('inf')})
background = select_naive_background_windows(events, window_duration=5.0, hop_duration=2.5, event_buffer=10.0)
# fill in the remaining cols the same way to_annotations_schema does in the DCLDE build script
background['Duration'] = background['FileEndSec'] - background['FileBeginSec']
background['CenterTime'] = (background['FileBeginSec'] + background['FileEndSec']) / 2
background['Labels'] = 'Background'
background['BackgroundMethod'] = 'naive'
background['KW'] = 0
background[['FileOk', 'LocalFileOk']] = True
background[['HasQ', 'CalltypeHasQ', 'EcotypeCertain']] = False
background
DORI_ONC = pd.concat([DORI_ONC, background], ignore_index=True)
print(DORI_ONC['Labels'].value_counts(dropna=False))
DORI_ONC['uid'] = DORI_ONC.index
set(DCLDE.columns) - set(DORI_ONC.columns)

selecting background windows: 100%|██████████| 24136/24136 [00:17<00:00, 1417.76file/s]


Labels
Background    2286814
HW              18473
KW_und          16693
SRKW            12352
UndBio           6458
TKW              2873
NaN              2255
OKW               184
AB                112
NRKW               41
Name: count, dtype: int64


set()

In [214]:
# merge DCLDE and DORI_ONC; cols that only exist in DORI_ONC are NaN on the DCLDE rows
COMBINED = pd.concat([DCLDE, DORI_ONC], ignore_index=True)
# uid was the row index in each source, so the two overlap; reassign
COMBINED['uid'] = COMBINED.index
print(COMBINED.shape)
# COMBINED.groupby(['Provider', 'Labels'], dropna=False).size().unstack(0, fill_value=0)

/tmp/ipykernel_173886/55983249.py:2: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  COMBINED = pd.concat([DCLDE, DORI_ONC], ignore_index=True)


(3526442, 41)


In [215]:
COMBINED

,Soundfile,Dataset,LowFreqHz,HighFreqHz,FileEndSec,UTC,FileBeginSec,ClassSpecies,KW,KW_certain,...,depth,doi,species_label,species_label_clean,species_label_source,ecotype_label_source,call_annotation,call_annotation_source,sighting_annotation,sighting_annotation_source
0,DFOCRP.KkHK0R2F-NML1.SM2M-3.20131011_000600Z.flac,NorthBc,1062.50,1187.50,251.695,2013-10-11 00:10:11.439,251.439,AB,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,DFOCRP.KkHK0R2F-NML1.SM2M-3.20131011_000600Z.flac,NorthBc,1062.50,1187.50,268.256,2013-10-11 00:10:28,268.000,AB,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,DFOCRP.KkHK0R2F-NML1.SM2M-3.20131011_000600Z.flac,NorthBc,1093.75,1218.75,297.952,2013-10-11 00:10:57.6,297.600,AB,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,DFOCRP.KkHK0R2F-NML1.SM2M-3.20131011_002100Z.flac,NorthBc,1000.00,1218.75,26.624,2013-10-11 00:21:26.176,26.176,AB,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,DFOCRP.KkHK0R2F-NML1.SM2M-3.20131011_002100Z.flac,NorthBc,1062.50,1250.00,40.895,2013-10-11 00:21:40.639,40.639,AB,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3526437,JASCOAMARHYDROPHONEE000186_20221229T190808.505...,DORI VENUS (Strait of Georgia),NaN,NaN,287.500,NaN,282.500,NaN,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3526438,JASCOAMARHYDROPHONEE000186_20221229T190808.505...,DORI VENUS (Strait of Georgia),NaN,NaN,290.000,NaN,285.000,NaN,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3526439,JASCOAMARHYDROPHONEE000186_20221229T190808.505...,DORI VENUS (Strait of Georgia),NaN,NaN,292.500,NaN,287.500,NaN,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3526440,JASCOAMARHYDROPHONEE000186_20221229T190808.505...,DORI VENUS (Strait of Georgia),NaN,NaN,295.000,NaN,290.000,NaN,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
